# IPL Auction Arena: cloud GPU runner (Kaggle)

Runs the auction engine with **local Ollama on a free Kaggle GPU**, so LLM bidders answer in about a second
instead of 20+ seconds on the laptop. Step-by-step guide: `docs/kaggle-gpu.md` in the repo.

**Before you press Run All, check the notebook settings (right panel):**
1. *Accelerator*: **GPU T4 x2** (or P100)
2. *Internet*: **On** (needs a phone-verified Kaggle account)
3. *Persistence*: Files only (optional)

Nothing private goes up here: the notebook clones the **public** repo only. `private/` never leaves the laptop.

In [ ]:
# 1. Is there a GPU?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Settings (change MODEL here; use the same model on the laptop so dev and cloud runs match)
import os
REPO   = "https://github.com/HarshThumar/ipl-auction-arena.git"
BRANCH = "master"
MODEL  = "qwen2.5:7b"          # docs name Qwen2.5-7B or Llama-3.1-8B ("llama3.1:8b")
WORK   = "/kaggle/working/arena"
os.environ["OLLAMA_MODEL"] = MODEL
os.environ["OLLAMA_HOST"]  = "http://127.0.0.1:11434"

In [ ]:
# 3. Install Ollama (about 1 minute)
!apt-get update -qq || true
!apt-get install -qq -y zstd pciutils || true
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# 4. Start the Ollama server in the background and wait until it answers
import subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434",
           OLLAMA_NUM_PARALLEL="1",      # one request at a time: reproducible order (failure register A15)
           OLLAMA_KEEP_ALIVE="24h")      # keep the model loaded between calls
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2).read().decode()); break
    except Exception:
        time.sleep(1)
else:
    print(open("/tmp/ollama.log").read()[-2000:]); raise SystemExit("ollama did not start")

In [ ]:
# 5. Download the model (about 5 GB, 2-4 minutes; it is downloaded again in every new session)
!ollama pull {MODEL} 2>&1 | tail -n 2
!ollama list

In [ ]:
# 6. Get the code (public repo) and install its requirements
!rm -rf {WORK} && git clone -q --depth 1 -b {BRANCH} {REPO} {WORK}
%cd {WORK}
!pip -q install -r requirements.txt
!git log --oneline -1

In [ ]:
# 7. Optional: reuse LLM answers from earlier sessions. Attach your saved "arena-llm-cache" dataset
#    (Add Input -> Your Datasets); any llm_cache folder found under /kaggle/input is copied in.
import glob, shutil
for src in glob.glob("/kaggle/input/**/llm_cache", recursive=True):
    shutil.copytree(src, "runs/llm_cache", dirs_exist_ok=True)
    print("restored cache from", src)
print("cached answers:", len(glob.glob("runs/llm_cache/*.json")))

In [ ]:
# 8. Sanity check: the engine tests (about 30 s)
!python -m pytest -q -x 2>&1 | tail -n 3

In [ ]:
# 9. Speed test: seconds per bidder decision on this GPU
!python scripts/bench_llm.py --n 5 --prompt-tokens 1500

In [ ]:
# 10. A full auction with code bidders (no LLM) to check everything runs
!python scripts/run_auction.py --season 2026 --seed 1 --bidders valuation | tail -n 8

## Your bidder

Once your LLM bidder exists (for example `src/bidders/my_bidder.py` with class `MyBidder`, pushed to GitHub),
run it here. It plays the teams you list; the others use the code bidders. Each seed is one auction;
every LLM answer is cached in `runs/llm_cache/`, so re-running a seed costs nothing and gives the same log.

In [ ]:
# 11. Run your bidder over several seeds (edit BIDDER and TEAMS, then run)
BIDDER = "src.bidders.stub_bidder:MyBidder"     # change to your class
TEAMS  = ["KKR"]                                 # or leave empty for all ten teams
SEEDS  = range(1, 4)
team_args = " ".join(f"--team {t}" for t in TEAMS)
for s in SEEDS:
    !python scripts/run_auction.py --season 2026 --seed {s} --bidders valuation --bidder {BIDDER} {team_args} | tail -n 6

In [ ]:
# 12. Save the results: event logs + LLM cache -> /kaggle/working/arena_runs.zip
#     Download it from the Output tab, or "Save Version" and turn the output into a dataset
#     ("arena-llm-cache") to reuse the cache next session (step 7).
import shutil
shutil.make_archive("/kaggle/working/arena_runs", "zip", WORK, "runs")
print(os.path.getsize("/kaggle/working/arena_runs.zip") // 1024, "KB")
